### Objective:

##### Transform the Silver-layer YouTube trending data into business-ready Gold tables by aggregating key metrics such as views, likes, engagement rates, category performance, country performance, channel analytics, upload timing insights, and top trending videos to support reporting and dashboarding.

#### Read Silver Layer

In [0]:
silver_path = "/Volumes/workspace/default/youtube_data/silver"

silver_df = spark.read \
.format("delta") \
.load(silver_path)

display(silver_df.limit(10))

video_id,trending_date,title,channel_title,category_id,publish_time,tags,views,likes,dislikes,comment_count,thumbnail_link,comments_disabled,ratings_disabled,video_error_or_removed,description,country,publish_year,publish_month,publish_day,publish_hour,engagement_rate,like_percentage
1UE5Dq1rvUA,17.14.11,Taylor Swift Perform Ready For It - SNL,Ken Reactz,24,2017-11-12T05:18:02.000Z,[none],320964,8069,285,717,https://i.ytimg.com/vi/1UE5Dq1rvUA/default.jpg,False,False,False,Thanks for watching please subscribe and subscribe like this video\n\nFollow me on ig @fuego_ken,CA,2017,11,12,5,0.027373786468264355,2.513989107812714
1KHCqtTC0EQ,17.14.11,"Toronto Raptors vs Boston Celtics: November 12, 2017",Motion Station,17,2017-11-12T23:20:25.000Z,"""sp:ty=high""|""sp:dt=2017-11-12T15:30:00-05:00""|""sp:vl=en-US""|""sp:st=basketball""|""sp:li=nba""|""sp:ti:home=BOS""|""sp:ti:away=TOR""",102766,294,39,39,https://i.ytimg.com/vi/1KHCqtTC0EQ/default.jpg,False,False,False,Al Horford scores 21 points as the Boston Celtics beat the Toronto Raptors 95-94,CA,2017,11,12,23,0.0032403713290387872,0.28608683805928026
GjP_A1Q1sjM,17.14.11,川普离京后立马“翻脸” 中国发布“惊人”政策（《万维读报》20171110）,万维TV,22,2017-11-11T02:41:22.000Z,"""creaders""|""万维TV""|""万维读者网""|""万维读报""|""川普""|""习近平""|""双十一""|""马云""|""淘宝""|""天猫""|""沙特""|""王子""|""国王""|""台湾""|""解放军""|""特朗普""",320294,600,254,370,https://i.ytimg.com/vi/GjP_A1Q1sjM/default.jpg,False,False,False,台湾军事弱点大曝光；川普为何如此讨中国喜欢？,CA,2017,11,11,2,0.0030284675953967293,0.18732789249876675
cmoknv58jjE,17.16.11,FLIPPING OVER SUPERCAR! *GONE VERY WRONG*,Tanner Braungardt,22,2017-11-14T21:00:02.000Z,"""tanner""|""tanner braungardt""|""trampoline tricks""|""cliff jumping""|""teaching a backflip""|""flip over car""|""flipping over supercar""|""flip over car gone wrong""|""car flip accident""|""horrible accident""|""near death experience""|""near death fail""|""car fail caught on camera""|""tumbling fails""|""gymnastic fails""|""worst fails compilation""|""epic fail compilation""|""audi r8 crash""|""supercar crash compilation""|""supercar crash""|""luxary car crash""|""redbull""|""parkour fail""|""parkour and freerunning""",1803724,57470,5261,7483,https://i.ytimg.com/vi/cmoknv58jjE/default.jpg,False,False,False,"Soooo that happened...\n\nFLIPPER- https://www.instagram.com/bagels_payne/\n\nhttps://www.youtube.com/user/baileypayne12\n\nCAR/DRIVER/SCARED CHILD- https://www.instagram.com/tannerbraungardt/\n\nJACK (BROTHER)- https://www.instagram.com/jackthepayne/\n\nhttps://www.youtube.com/channel/UCmbTQB6UFr2AbPs0g0FmruA\n\nDOM- https://www.instagram.com/domitrick/\n\nhttps://www.youtube.com/channel/UCS3-mVxKDzUucEYD2gYTnGw/featured\n\nDon't forget to leave a LIKE and SUBSCRIBE - http://bit.ly/subTannerBraungardt - if you enjoyed! Also, SHARE with your friends! \n\nWATCH MORE:\n\nTRAMPOLINE VS - https://www.youtube.com/playlist?list=PLfISKSyDz8gy8BRM1-OBoLKl92z_xfQ6g\nCHALLENGES - https://www.youtube.com/playlist?list=PLfISKSyDz8gxitc78S6EFA7-WpklbHP_B\nLATEST - https://www.youtube.com/playlist?list=PLfISKSyDz8gxRFITdSH-lpVGQ1Pnc0YXl\n\nFOLLOW ME:\nInstagram: https://instagram.com/tannerbraungardt/\nSnapchat: http://snapchat.com/add/tanner_b24\nTwitter: https://twitter.com/braungardtanner\nFacebook: https://facebook.com/tannerbraungardt/\n\nOfficial Merch:\nhttp://www.tbraungardt.com/\n\nBusiness Inquiries: braungardtbiz@gmail.com\n\nFAN MAIL ADDRESS: \nTanner Braungardt\nP.O. Box 98\nAugusta, KS 67010\n\nSkybound's Amazon: http://amzn.to/2bWpQlc\nSkyBound Stratos - 10% Off\nUse Code: TANNER10 http://bit.ly/2cOikfQ\nSkyBoundUSA.com - http://bit.ly/2cr7QSl\n\nOutro music: Zara Larsson - Ain't My Fault (R3hab Remix)\n\nIf you've read all of this I love you\n\n-----------------------------------------------------------------------------------------------",CA,2017,11,14,21,0.03601049828022469,3.18618591314414
dq6G2YWoRqA,17.16.11,OrelSan - Tout va bien [CLIP OFFICIEL],orelsan,10,2017-11-15T12:07:54.000Z,[none],1267473,111445,1663,5994,https://i.ytimg.co

#### GOLD TABLE 1: Top Channels Analytics

##### Which channels generate the highest views and engagement?

In [0]:
from pyspark.sql.functions import sum, avg, count

top_channels = silver_df.groupBy(
    "channel_title"
).agg(
    count("*").alias("total_trending_videos"),
    sum("views").alias("total_views"),
    sum("likes").alias("total_likes"),
    avg("engagement_rate").alias("avg_engagement_rate")
).orderBy(
    "total_views",
    ascending=False
)

In [0]:
display(top_channels.limit(20))

channel_title,total_trending_videos,total_views,total_likes,avg_engagement_rate
T-Series,92,393615082,8481916,0.0301717862694249
Marvel Entertainment,57,339531740,15183317,0.04949932700131736
ibighit,26,304432938,47903227,0.21853633365981695
WWE,208,296786710,5492670,0.024382803203538382
5-Minute Crafts,165,291587587,3148584,0.012173714186890961
The Late Show with Stephen Colbert,271,271004192,3812896,0.016488908549704116
Dude Perfect,39,253210259,11773787,0.05379279210902837
PewDiePie,93,252005593,21827216,0.09909677660132898
TheEllenShow,231,244538782,6760970,0.028274691690756706
20th Century Fox,52,203523518,4973549,0.023094489295097047


#### GOLD TABLE 2: Country Performance Analytics

##### Which countries have the most engagement?

In [0]:
country_performance = silver_df.groupBy(
    "country"
).agg(
    count("*").alias("total_videos"),
    sum("views").alias("total_views"),
    sum("likes").alias("total_likes"),
    avg("engagement_rate").alias("avg_engagement_rate")
)

In [0]:
display(country_performance)

country,total_videos,total_views,total_likes,avg_engagement_rate
GB,3272,4227437526,177730286,0.05129089844019688
US,6351,4815388944,219068662,0.045927618792569735
CA,24427,11872789720,449417025,0.04054089965068233
IN,16307,6486515520,170903637,0.02452854465356162


#### GOLD TABLE 3: Category Performance Analytics

##### Which content categories trend the most?

In [0]:
category_mapping = {
    "1": "Film & Animation",
    "2": "Autos & Vehicles",
    "10": "Music",
    "15": "Pets & Animals",
    "17": "Sports",
    "18": "Short Movies",
    "19": "Travel & Events",
    "20": "Gaming",
    "21": "Videoblogging",
    "22": "People & Blogs",
    "23": "Comedy",
    "24": "Entertainment",
    "25": "News & Politics",
    "26": "Howto & Style",
    "27": "Education",
    "28": "Science & Technology",
    "29": "Nonprofits & Activism",
    "30": "Movies",
    "31": "Anime/Animation",
    "32": "Action/Adventure",
    "33": "Classics",
    "34": "Comedy",
    "35": "Documentary",
    "36": "Drama",
    "37": "Family",
    "38": "Foreign",
    "39": "Horror",
    "40": "Sci-Fi/Fantasy",
    "41": "Thriller",
    "42": "Shorts",
    "43": "Shows",
    "44": "Trailers",
}

###### Step 1: Create Category Mapping DataFrame

In [0]:
from pyspark.sql import Row

category_df = spark.createDataFrame([
    Row(category_id=k, category_name=v)
    for k, v in category_mapping.items()
])

###### Step 2: Create Category Performance Table

In [0]:
from pyspark.sql.functions import count, sum, avg

category_performance = silver_df.groupBy(
    "category_id"
).agg(
    count("*").alias("total_videos"),
    sum("views").alias("total_views"),
    avg("engagement_rate").alias("avg_engagement_rate")
)

###### Step 3: Join Category Names

In [0]:
from pyspark.sql.functions import col

category_performance = category_performance.withColumn(
    "category_id",
    col("category_id").cast("string")
)

###### Join:

In [0]:
category_performance = category_performance.join(
    category_df,
    on="category_id",
    how="left"
)

###### Step 4: Reorder Columns

In [0]:
category_performance = category_performance.select(
    "category_name",
    "category_id",
    "total_videos",
    "total_views",
    "avg_engagement_rate"
)

###### Step 5: Sort for Display

In [0]:
display(
    category_performance.orderBy(
        "total_views",
        ascending=False
    )
)

category_name,category_id,total_videos,total_views,avg_engagement_rate
Entertainment,24,18284,8550654374,0.026397681035632403
Music,10,4452,6698486004,0.06042896033306055
Comedy,23,3813,2168619110,0.06081350400152354
Sports,17,2942,1884733282,0.019597352317765875
People & Blogs,22,4574,1822294101,0.03554364545183982
Film & Animation,1,2186,1536302866,0.03348493530888966
News & Politics,25,6079,1316171881,0.02302671869324571
Howto & Style,26,2534,1180227238,0.05352935840728016
Science & Technology,28,1359,775999324,0.062286718262748904
Gaming,20,1068,640627185,0.05813799370831839


#### GOLD TABLE 4: Upload Time Analytics

##### What are the optimal upload hours based on views and engagement?

In [0]:
upload_time_analytics = silver_df.groupBy(
    "publish_hour"
).agg(
    count("*").alias("video_count"),
    avg("views").alias("avg_views"),
    avg("engagement_rate").alias("avg_engagement")
).orderBy(
    "avg_views",
    ascending=False
)

In [0]:
display(upload_time_analytics)

publish_hour,video_count,avg_views,avg_engagement
5,1916,814586.4211899791,0.02985104844393782
4,1845,800307.7658536586,0.030624482257371428
9,1335,738260.397752809,0.033823298452023205
7,1466,655106.102319236,0.02437097653345193
13,2526,636927.524148852,0.036701273454770554
8,1551,613223.6047711154,0.029119407713336622
6,1490,600949.2946308724,0.02422531560234826
12,2144,549302.5550373135,0.03584723627640009
14,3120,543434.4753205128,0.03478968049257813
15,3302,535853.4727437916,0.03956531495209295


#### GOLD TABLE 5: Top Trending Videos

##### Which videos achieved maximum reach?

In [0]:
top_videos = silver_df.select(
    "video_id",
    "title",
    "channel_title",
    "country",
    "views",
    "likes",
    "engagement_rate"
).orderBy(
    "views",
    ascending=False
)

In [0]:
display(top_videos.limit(20))

video_id,title,channel_title,country,views,likes,engagement_rate
TyHvyGVs42U,"Luis Fonsi, Demi Lovato - Échame La Culpa",LuisFonsiVEVO,GB,143408235,2686169,0.01973656533740897
zEf423kYfqk,"Becky G, Natti Natasha - Sin Pijama (Official Video)",BeckyGVEVO,GB,88568646,1185357,0.014176563114671529
-BQJo3vK8O8,Maluma - El Préstamo (Official Video),MalumaVEVO,US,48431654,609101,0.013178839607666507
WtE011iVx1Q,"Sebastián Yatra - Por Perro ft. Luis Figueroa, Lary Over",SebastianYatraVEVO,GB,47669287,396337,0.008649007064024264
Ck4xHocysLw,Ozuna - Única (Video Oficial) 🐻 A U R A,Ozuna,GB,42923278,495422,0.011963508472023036
7C2z4GqqS5E,BTS (방탄소년단) 'FAKE LOVE' Official MV,ibighit,US,39349927,3880071,0.11619782674565063
7C2z4GqqS5E,BTS (방탄소년단) 'FAKE LOVE' Official MV,ibighit,CA,39349927,3880074,0.11619805546272043
7C2z4GqqS5E,BTS (방탄소년단) 'FAKE LOVE' Official MV,ibighit,GB,39349927,3880074,0.11619805546272043
VTzD0jNdrmo,Rkm & Ken-Y ❌ Natti Natasha - Tonta [Official Video],Pina Records,GB,39118664,383030,0.010132963641089583
i0p1bmr0EmE,TWICE What is Love? M/V,jypentertainment,GB,38873543,1111595,0.033910827217369925


#### Save Gold Tables

In [0]:
top_channels.write \
.format("delta") \
.mode("overwrite") \
.save("/Volumes/workspace/default/youtube_data/gold_top_channels")

country_performance.write \
.format("delta") \
.mode("overwrite") \
.save("/Volumes/workspace/default/youtube_data/gold_country_performance")

category_performance.write \
.format("delta") \
.mode("overwrite") \
.save("/Volumes/workspace/default/youtube_data/gold_category_performance")

upload_time_analytics.write \
.format("delta") \
.mode("overwrite") \
.save("/Volumes/workspace/default/youtube_data/gold_upload_time")

top_videos.write \
.format("delta") \
.mode("overwrite") \
.save("/Volumes/workspace/default/youtube_data/gold_top_videos")

print("All Gold tables created successfully!")

All Gold tables created successfully!


#### Verify Gold Tables

In [0]:
display(top_channels.limit(10))
display(country_performance)
display(category_performance)
display(upload_time_analytics)
display(top_videos.limit(10))

channel_title,total_trending_videos,total_views,total_likes,avg_engagement_rate
T-Series,92,393615082,8481916,0.0301717862694249
Marvel Entertainment,57,339531740,15183317,0.04949932700131736
ibighit,26,304432938,47903227,0.21853633365981695
WWE,208,296786710,5492670,0.024382803203538382
5-Minute Crafts,165,291587587,3148584,0.012173714186890961
The Late Show with Stephen Colbert,271,271004192,3812896,0.016488908549704116
Dude Perfect,39,253210259,11773787,0.05379279210902837
PewDiePie,93,252005593,21827216,0.09909677660132898
TheEllenShow,231,244538782,6760970,0.028274691690756706
20th Century Fox,52,203523518,4973549,0.023094489295097047


country,total_videos,total_views,total_likes,avg_engagement_rate
GB,3272,4227437526,177730286,0.05129089844019688
US,6351,4815388944,219068662,0.045927618792569735
CA,24427,11872789720,449417025,0.04054089965068233
IN,16307,6486515520,170903637,0.02452854465356162


category_name,category_id,total_videos,total_views,avg_engagement_rate
Comedy,23,3813,2168619110,0.06081350400152354
Film & Animation,1,2186,1536302866,0.03348493530888966
News & Politics,25,6079,1316171881,0.02302671869324571
Science & Technology,28,1359,775999324,0.062286718262748904
Entertainment,24,18284,8550654374,0.026397681035632403
Travel & Events,19,276,69129575,0.03966851550109116
Gaming,20,1068,640627185,0.05813799370831839
Pets & Animals,15,391,136449826,0.05636207837012476
Nonprofits & Activism,29,139,50858714,0.0441562971696167
Autos & Vehicles,2,365,126822299,0.03669947631152415


publish_hour,video_count,avg_views,avg_engagement
5,1916,814586.4211899791,0.02985104844393782
4,1845,800307.7658536586,0.030624482257371428
9,1335,738260.397752809,0.033823298452023205
7,1466,655106.102319236,0.02437097653345193
13,2526,636927.524148852,0.036701273454770554
8,1551,613223.6047711154,0.029119407713336622
6,1490,600949.2946308724,0.02422531560234826
12,2144,549302.5550373135,0.03584723627640009
14,3120,543434.4753205128,0.03478968049257813
15,3302,535853.4727437916,0.03956531495209295


video_id,title,channel_title,country,views,likes,engagement_rate
TyHvyGVs42U,"Luis Fonsi, Demi Lovato - Échame La Culpa",LuisFonsiVEVO,GB,143408235,2686169,0.01973656533740897
zEf423kYfqk,"Becky G, Natti Natasha - Sin Pijama (Official Video)",BeckyGVEVO,GB,88568646,1185357,0.014176563114671529
-BQJo3vK8O8,Maluma - El Préstamo (Official Video),MalumaVEVO,US,48431654,609101,0.013178839607666507
WtE011iVx1Q,"Sebastián Yatra - Por Perro ft. Luis Figueroa, Lary Over",SebastianYatraVEVO,GB,47669287,396337,0.008649007064024264
Ck4xHocysLw,Ozuna - Única (Video Oficial) 🐻 A U R A,Ozuna,GB,42923278,495422,0.011963508472023036
7C2z4GqqS5E,BTS (방탄소년단) 'FAKE LOVE' Official MV,ibighit,GB,39349927,3880074,0.11619805546272043
7C2z4GqqS5E,BTS (방탄소년단) 'FAKE LOVE' Official MV,ibighit,US,39349927,3880071,0.11619782674565063
7C2z4GqqS5E,BTS (방탄소년단) 'FAKE LOVE' Official MV,ibighit,CA,39349927,3880074,0.11619805546272043
VTzD0jNdrmo,Rkm & Ken-Y ❌ Natti Natasha - Tonta [Official Video],Pina Records,GB,39118664,383030,0.010132963641089583
i0p1bmr0EmE,TWICE What is Love? M/V,jypentertainment,US,38873543,1111592,0.033910569973001944
